# Minimum-intervention sweep at N=12 (w=50)

**Purpose.** Map the rescue boundary in (n_pde, n_data) space. We know
three single-axis rescues work (width, n_pde=160k, n_data≥500). This asks:
can we trade off n_pde and n_data against each other?

**Why this is a rerun.** Broken pipeline had *every* config failing at 86–92%,
including configs that were already proven to work. Redone on canonical pipeline.

**Approx runtime:** ~8 h on Kaggle T4. Configs ordered cheap→expensive so
a session crash still leaves useful data.

In [ ]:
import os, sys, json, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch

for p in ["/kaggle/input/kdv-core", "/kaggle/working", "."]:
    if (Path(p) / "kdv_core.py").exists():
        sys.path.insert(0, p)
        break
import kdv_core as K
print(f"device = {K.DEVICE}")

OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUT.mkdir(parents=True, exist_ok=True)
print(f"output dir = {OUT}")


In [ ]:
K.quick_sanity_check(seed=99, adam_iter=500)

### Configurations (cheap → expensive)

In [ ]:
N = 12
WIDTH = 50
SEED = 99

CONFIGS = [
    # (name,                  n_pde,   n_data,  note)
    ("D_lowpde_lowdata",      20_000,    400,   "sanity - both low, expect FAIL"),
    ("A_lowpde_highdata",     20_000,  2_000,   "extreme data, minimal collocation"),
    ("E_midpde_highdata",     40_000,  2_000,   "probe boundary"),
    ("C_midpde_middata",      80_000,    500,   "mid + mid"),
    ("F_midpde_highdata2",    80_000,  1_000,   "probe boundary"),
    ("B_highpde_lowdata",    160_000,    400,   "extreme collocation, minimal data"),
]

print(f"{'name':<25} {'n_pde':>8} {'n_data':>7}  note")
for name, npde, ndata, note in CONFIGS:
    print(f"  {name:<25} {npde:>8,} {ndata:>7,}  {note}")


### Run the sweep

In [ ]:
cfg = K.make_config(N)
K.print_config(cfg)

results = []
for name, npde, ndata, note in CONFIGS:
    tag = f"N{N}_w{WIDTH}_{name}"
    ckpt_path = OUT / f"checkpoint_{tag}.pt"

    if ckpt_path.exists():
        print(f"\n[resume] {tag} done, loading.")
        ck = K.load_checkpoint(ckpt_path)
        hist = ck["history"]
        results.append(dict(name=name, n_pde=npde, n_data=ndata, note=note,
                            L2_adam=hist["adam_l2"], L2_final=hist["lbfgs_l2"],
                            total_min=(hist["adam_time"]+hist["lbfgs_time"])/60))
        continue

    print("\n" + "=" * 70)
    print(f"{tag}  |  n_pde={npde:,}  n_data={ndata:,}")
    print(f"  note: {note}")
    print("=" * 70)
    K.set_seed(SEED)
    batch = K.build_data(cfg, seed=SEED, n_pde=npde, n_data=ndata)
    m = K.PINN(width=WIDTH).to(K.DEVICE)
    print(f"  params = {m.n_params():,}")
    t0 = time.time()
    hist = K.train(m, batch, adam_iter=15000, lbfgs_iter=2000)
    print(f"  total {(time.time()-t0)/60:.1f} min")
    K.save_checkpoint(ckpt_path, m, hist, cfg,
                      extras=dict(tag=tag, name=name, n_pde=npde, n_data=ndata, seed=SEED))
    print(f"  saved {ckpt_path}")

    results.append(dict(name=name, n_pde=npde, n_data=ndata, note=note,
                        L2_adam=hist["adam_l2"], L2_final=hist["lbfgs_l2"],
                        total_min=(hist["adam_time"]+hist["lbfgs_time"])/60))
    pd.DataFrame(results).to_csv(OUT / "min_intervention_results.csv", index=False)

df = pd.DataFrame(results)
df["rescued"] = df["L2_final"] < 5.0
print("\n--- Results ---")
print(df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
df.to_csv(OUT / "min_intervention_results.csv", index=False)


### Combine with known prior anchors

In [ ]:
KNOWN = pd.DataFrame([
    dict(name="baseline_FAIL",           n_pde=56_666,  n_data=1_133, L2_final=42.1502, rescued=False),
    dict(name="rescue_pde160k_default",  n_pde=160_000, n_data=1_133, L2_final=2.4788,  rescued=True),
    dict(name="rescue_ndata500_default", n_pde=56_666,  n_data=500,   L2_final=1.9530,  rescued=True),
    dict(name="rescue_ndata2000_default",n_pde=56_666,  n_data=2_000, L2_final=1.2219,  rescued=True),
    dict(name="fail_pde40k_default",     n_pde=40_000,  n_data=1_133, L2_final=42.8802, rescued=False),
    dict(name="fail_pde80k_default",     n_pde=80_000,  n_data=1_133, L2_final=24.1723, rescued=False),
])
KNOWN["source"] = "prior runs"
df_new = df[["name","n_pde","n_data","L2_final","rescued"]].copy()
df_new["source"] = "this notebook"
full = pd.concat([KNOWN, df_new], ignore_index=True)
full.to_csv(OUT / "rescue_map_full.csv", index=False)
print(full[["name","n_pde","n_data","L2_final","rescued","source"]]
      .to_string(index=False, float_format=lambda v: f"{v:.4f}"))


### Rescue map plot

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
for _, row in full.iterrows():
    color  = "green" if row["rescued"] else "red"
    marker = "s" if row["source"] == "this notebook" else "o"
    ax.scatter(row["n_pde"], row["n_data"], s=200, c=color,
               marker=marker, edgecolors="k", linewidths=1.2, zorder=3)
    ax.annotate(f"{row['name']}\n{row['L2_final']:.2f}%",
                (row["n_pde"], row["n_data"]),
                textcoords="offset points", xytext=(8, 8), fontsize=7)

ax.scatter([],[],s=150,c="green",marker="o",edgecolors="k",label="rescued (L2<5%)")
ax.scatter([],[],s=150,c="red",  marker="o",edgecolors="k",label="failed")
ax.scatter([],[],s=150,c="grey", marker="o",edgecolors="k",label="prior runs", alpha=0.6)
ax.scatter([],[],s=150,c="grey", marker="s",edgecolors="k",label="this notebook")

ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("n_pde (collocation points)"); ax.set_ylabel("n_data (supervised points)")
ax.set_title(f"N={N} rescue map (w={WIDTH}) — minimum sufficient intervention")
ax.legend(loc="lower left"); ax.grid(alpha=0.3, which="both")
fig.tight_layout()
fig.savefig(OUT / "rescue_map.png", dpi=140, bbox_inches="tight")
plt.show()


### Interpretation

In [ ]:
print("\n" + "=" * 65)
print("RESCUE SUMMARY (sorted by L2)")
print("=" * 65)
for _, r in full.sort_values("L2_final").iterrows():
    flag = "RESCUED" if r["rescued"] else "failed "
    print(f"  {flag}  n_pde={int(r['n_pde']):>7,}  n_data={int(r['n_data']):>6,}"
          f"  L2={r['L2_final']:6.2f}%  ({r['name']})")

rescued_subset = full[full["rescued"]]
if len(rescued_subset):
    cheapest = rescued_subset.copy()
    cheapest["total_pts"] = cheapest["n_pde"] + cheapest["n_data"]
    best = cheapest.sort_values("total_pts").iloc[0]
    print(f"\nCheapest rescue (by total points): {best['name']}")
    print(f"  n_pde={int(best['n_pde']):,}  n_data={int(best['n_data']):,}  "
          f"total={int(best['total_pts']):,}  L2={best['L2_final']:.2f}%")
print("\nDone. Outputs in", OUT)
